# Source verification, labels, quality and repeatability
Revised replacement for the historical conversion notebook. The original source-to-site export remains in Git history; existing NPZ files retain their patient/site identifiers. Source verification and extraction now use scripts/prepare_revision_source.py through the canonical preflight.

Run make revision-preflight first (or the explicitly enabled stages in Revision-Evaluation.ipynb). This notebook **reads saved audits only**: no normalization, imputation, feature selection or model fitting occurs here. Never infer independence from the number of sites.


In [ ]:
from pathlib import Path
import json
import os
import sys
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "run_revision.py").is_file()
)
sys.path.insert(0, str(ROOT))


def read_json(path):
    return json.loads(Path(path).read_text())


def table(path, **kwargs):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"**Unavailable:** {path.relative_to(ROOT)}"))
        return pd.DataFrame()
    try:
        return pd.read_csv(path, **kwargs)
    except pd.errors.EmptyDataError:
        display(Markdown(f"**No estimable rows:** {path.relative_to(ROOT)}"))
        return pd.DataFrame()


AUDIT = ROOT / os.environ.get("IIVV_AUDIT", "results/revision-expanded-preflight")
manifest = read_json(AUDIT / "run_manifest.json")
if manifest.get("synthetic") or manifest.get("status") not in {
    "preflight_passed",
    "completed",
}:
    raise ValueError("Use successful real-data preflight/completed outputs.")
SOURCE = Path(manifest["arguments"]["source_output"])
SOURCE = SOURCE if SOURCE.is_absolute() else ROOT / SOURCE
display(
    {
        key: manifest.get(key)
        for key in ["status", "input_summary", "source_input_hashes"]
    }
)

## Explicit label semantics — R2 C3; R1 C15–16
Cancer and high risk are the positive binary classes. Moderate/severe dysplasia and CIS belong to the high-risk group. Unmapped diagnoses are excluded explicitly. This reproduces the audited export semantics; it does not establish mixed-footprint pathology adjudication.


In [ ]:
from revision.data import PATHOLOGY, CLASS_NAMES

display(
    pd.DataFrame(
        sorted(PATHOLOGY.items()), columns=["raw_pathology", "canonical_class"]
    )
)
display(
    pd.DataFrame(
        [
            {
                "task": task,
                "class_index": i,
                "class_name": label,
                "binary_positive": len(labels) == 2 and i == 1,
            }
            for task, labels in CLASS_NAMES.items()
            for i, label in enumerate(labels)
        ]
    )
)

## Source alignment and available clinical fields — R1 C3, C11–14, C24–25; R2 C1–2
Source spectra and cohort/case pairs are verified before linking metadata or test means. Inspect unavailable fields and opaque MATLAB string fields. HPV/grade payload integers are not Unicode text. Test means do not recover individual bursts, pressure control, histology sectioning or blinding.


In [ ]:
extraction = read_json(SOURCE / "source_extraction.json")
display(
    {
        key: extraction.get(key)
        for key in [
            "source_fields",
            "n_sites",
            "all_exported_zdata_and_cohort_case_pairs_verified",
            "repeat_units",
            "test_count_distribution",
            "unavailable_metadata",
            "undecoded_opaque_matlab_string_fields",
        ]
    }
)
display(read_json(AUDIT / "quality/clinical_metadata_status.json"))
display(read_json(AUDIT / "quality/source_quality_audit.json"))
display(table(AUDIT / "quality/source_quality_audit.csv").head())

## Cohort, exclusions and unequal sampling — R1 C10, M6–7; R3 C1–2
Inspect the actual unique-patient count rather than assuming the manuscript's historical count. Cohort tables contain patient-level identifiers; keep executed notebooks under the study's access conditions.


In [ ]:
sites = table(
    AUDIT / "quality/site_inclusion_audit.csv",
    dtype={"patient_id": str, "sample_id": str},
)
display(
    {
        "sites": len(sites),
        "patients": sites.patient_id.nunique(),
        "unmapped_sites": int(sites.pathology.isna().sum()),
    }
)
display(
    sites.groupby("pathology", dropna=False).agg(
        n_sites=("sample_id", "size"), n_patients=("patient_id", "nunique")
    )
)
display(sites.loc[sites.pathology.isna(), ["sample_id", "raw_pathology"]])
display(table(AUDIT / "quality/sites_per_patient.csv"))

## Missingness, thresholds and anatomical selection — R1 C2–5, C12–14
The acquisition rejection percentage and missing coordinates after averaging are distinct quantities. Retention tables describe the cohort; the threshold is selected inside training folds. Association tests and covariate tables are descriptive, with unavailable tests retained explicitly.


In [ ]:
for name in [
    "missing_patterns_by_class.csv",
    "retention_by_threshold_class.csv",
    "class_by_location.csv",
    "missing_patterns_by_class_location.csv",
    "quality_cluster_associations.csv",
    "numeric_covariates_by_class.csv",
    "covariate_cluster_associations.csv",
]:
    display(Markdown(f"### {name}"))
    display(table(AUDIT / "quality" / name))
display(read_json(AUDIT / "quality/missingness_encoding_audit.json"))

## Test-mean repeatability — R1 C24–25
Coordinate CV, scalar test-mean CV, repositioning SD and ICC are different endpoints. The balanced absolute-agreement ICC uses the first three test positions with complete-site exclusions and patient-cluster intervals. It is not within-burst repeatability or reliability of every classifier coordinate.


In [ ]:
display(read_json(AUDIT / "repeatability/status.json"))
display(table(AUDIT / "repeatability/icc_and_inter_site_variability.csv"))
display(table(AUDIT / "repeatability/site_repeatability.csv").describe(include="all"))

## Next step
After the server evaluation and completion check, use Figures-and-Tables.ipynb for all four tasks, held-out metrics, dysplasia errors, ablations, uncertainty, clinical operating points and computational requirements. Leave manuscript imports until review of the completed outputs.
